## Preparacion del entorno

Importacion de librerias necesarias para la evaluacion.

In [8]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

## Configuracion

Definicion de rutas y parametros para la evaluacion.

In [9]:
models_dir = Path('trained_models')
models = ['yolov8n', 'yolov8s', 'yolov8m']
class_names = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

test_csv = 'dataset/test_subset.csv'
test_dir = Path('dataset/test')
yaml_path = 'yolo_dataset/data.yaml'

results = {}

## Carga de modelos y evaluacion

Carga de los modelos entrenados y calculo de metricas principales.

In [10]:
for model_name in models:
    model_path = models_dir / f'{model_name}_best.pt'
    
    if not model_path.exists():
        continue
    
    model = YOLO(str(model_path))
    
    metrics = model.val(data=yaml_path, split='val')
    
    results[model_name] = {
        'metrics': metrics,
        'model': model
    }

Ultralytics 8.3.237 🚀 Python-3.12.1 torch-2.9.1+cu128 CPU (Intel Xeon Platinum 8370C CPU @ 2.80GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 13.6±17.2 MB/s, size: 408.1 KB)
val: Scanning /workspaces/DeepLearning/yolo_dataset/val/labels.cache... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 52.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 20% ━━────────── 1/5 13.8s/it 6.9s<55.4s


KeyboardInterrupt: 

## Predicciones individuales

Generacion de predicciones para cada imagen del conjunto de test.

In [ ]:
def get_predictions_per_class(model, test_dir, test_csv):
    df_test = pd.read_csv(test_csv)
    
    predictions = []
    ground_truth = []
    
    for _, row in df_test.iterrows():
        img_path = test_dir / (row['image'] + '.jpg')
        
        if img_path.exists():
            result = model(str(img_path), verbose=False)[0]
            
            if len(result.boxes) > 0:
                pred_class = int(result.boxes[0].cls[0])
                predictions.append(pred_class)
            else:
                predictions.append(-1)
            
            true_class = class_names.index(row['clase'])
            ground_truth.append(true_class)
    
    return np.array(predictions), np.array(ground_truth)

predictions_dict = {}
for model_name in models:
    if model_name in results:
        preds, gt = get_predictions_per_class(results[model_name]['model'], test_dir, test_csv)
        predictions_dict[model_name] = {'predictions': preds, 'ground_truth': gt}

## Matriz de confusion

Visualizacion de las matrices de confusion para cada modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Matrices de Confusion', fontsize=16)

for idx, model_name in enumerate(models):
    if model_name in predictions_dict:
        preds = predictions_dict[model_name]['predictions']
        gt = predictions_dict[model_name]['ground_truth']
        
        valid_idx = preds != -1
        preds_valid = preds[valid_idx]
        gt_valid = gt[valid_idx]
        
        cm = confusion_matrix(gt_valid, preds_valid)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names,
                    ax=axes[idx])
        axes[idx].set_title(f'{model_name}')
        axes[idx].set_ylabel('Etiqueta Real')
        axes[idx].set_xlabel('Etiqueta Predicha')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## Reporte de clasificacion

Metricas detalladas por clase para cada modelo.

In [ ]:
for model_name in models:
    if model_name in predictions_dict:
        preds = predictions_dict[model_name]['predictions']
        gt = predictions_dict[model_name]['ground_truth']
        
        valid_idx = preds != -1
        preds_valid = preds[valid_idx]
        gt_valid = gt[valid_idx]
        
        print(f"\n{'='*60}")
        print(f"Reporte de clasificacion - {model_name}")
        print(f"{'='*60}\n")
        print(classification_report(gt_valid, preds_valid, target_names=class_names))
        print(f"Accuracy: {accuracy_score(gt_valid, preds_valid):.4f}")

## Guardado de resultados

Exportacion de metricas a CSV para analisis posterior.

In [ ]:
results_data = []

for model_name in models:
    if model_name in results and model_name in predictions_dict:
        metrics = results[model_name]['metrics']
        preds = predictions_dict[model_name]['predictions']
        gt = predictions_dict[model_name]['ground_truth']
        
        valid_idx = preds != -1
        acc = accuracy_score(gt[valid_idx], preds[valid_idx])
        
        results_data.append({
            'Model': model_name,
            'mAP50': metrics.box.map50,
            'mAP50-95': metrics.box.map,
            'Precision': metrics.box.mp,
            'Recall': metrics.box.mr,
            'Accuracy': acc
        })

df_results = pd.DataFrame(results_data)
df_results.to_csv('model_results.csv', index=False)